In [1]:
import os
import numpy as np
import gym
import torch
import torch.nn as nn
import torchvision.transforms as T

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

# Reinforcement Learning - DQN을 사용한 Plane Shooting Game

In [26]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip -q install numpy==1.23.0
!pip -q install gym==0.25.2 pygame

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [3]:
!cp -r "/content/drive/MyDrive/fly ai/dataset/gym_examples" "/content/"

In [5]:
!cp /content/gym_examples/envs/reversi_random_template.py /content/gym_examples/envs/reversi_random.py
!ls /content/gym_examples/envs

__init__.py  reversi_random.py		 shooting_airplane.py
__pycache__  reversi_random_template.py


In [7]:
path = "/content/gym_examples/envs/shooting_airplane.py"
with open(path, "r") as f:
    code = f.read()
code = code.replace("import gymnasium as gym", "import gym")
code = code.replace("from gymnasium import spaces", "from gym import spaces")
code = code.replace("import gymnasium", "import gym")
with open(path, "w") as f:
    f.write(code)
print("Fixed shooting_airplane.py")

Fixed shooting_airplane.py


In [8]:
import sys
sys.path.insert(0, "/content")
# 혹시 이미 import된 모듈 제거
for m in list(sys.modules.keys()):
    if m.startswith("gym_examples"):
        del sys.modules[m]
import gym
import gym_examples

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment gym_examples/ShootingAirplane-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment gym_examples/Reversi-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [9]:
from gym.envs.registration import register

try:

    register(

        id="ShootingAirplane-v0",

        entry_point="gym_examples.envs.shooting_airplane:ShootingAirplaneEnv",

    )

    print("Environment registered.")

except Exception as e:

    print("Register skipped or already registered:", e)

env = gym.make("ShootingAirplane-v0", render_mode="text")

print(env)

print(env.action_space)

print(type(env.action_space))

Environment registered.
<OrderEnforcing<StepAPICompatibility<PassiveEnvChecker<ShootingAirplaneEnv<ShootingAirplane-v0>>>>>
MultiDiscrete([8 8])
<class 'gym.spaces.multi_discrete.MultiDiscrete'>


/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:542: UserWarning: WARN: Overriding environment ShootingAirplane-v0
  logger.warn(f"Overriding environment {spec.id}")
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


## 변수 정의

In [10]:
# Configuration parameters for the whole setup

seed = 42
gamma = 0.99    # Discount factor for past rewards
epsilon = 1.0    # Epsilon greedy parameter
epsilon_min = 0.1   # Minimum epsilon greedy parameter
epsilon_max = 1.0   # Maximum epsilon greedy parameter
epsilon_interval = epsilon_max - epsilon_min    # Rate at which to reduce chance of random action being taken
batch_size = 16     # Size of batch taken from replay buffer
max_steps_per_episode = 600
max_episodes = 5000

## 게임 환경 설정

상태 (state) 정의

- 보드판의 모양 : 8 x 8 행렬 * 3 채널
- 채널 0 : unseen
- 채널 1 : hit
- 채널 2 : miss

액션 정의
- 돌의 가능한 위치 8*8 = 64

In [11]:
env = gym.make("ShootingAirplane-v0", render_mode="text")
print(env)
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)

<OrderEnforcing<StepAPICompatibility<PassiveEnvChecker<ShootingAirplaneEnv<ShootingAirplane-v0>>>>>
action_space: MultiDiscrete([8 8])
observation_space: Box(0, 255, (8, 8, 1), uint8)


### env에서 정의한 action_space, observation_space 의 모양 확인

- action_space : 3개의 값의 튜플 (벡터)
- observation_space : HWC 형태의 이미지 (마지막 축이 단일 값인 15x15x1 텐서 -> pytorch를 사용할 경우 적절이 1x15x15 텐서로 수정 필요)

In [12]:
env.action_space.shape, env.observation_space.shape

((2,), (8, 8, 1))

### 네트워크 정의하기

참고 : Conv2d 파라미터

- in_channels(int) : Number of channels in the input image
- out_channels(int) : Number of channels produced by the convolution
- kernel_size(int or tuple) : Size of the convolving kernel
- stride(int or tuple, optional) : Stride of the convolution. Default 1
- padding(int, tuple or str, optional) : Padding added to all four sides of the input. Default 0
- padding_mode(str, optional) : 'zeros', 'reflect', 'replicate' or 'circular'. Default 'zeros'

In [15]:
num_actions = 64

class QModel(nn.Module):
  def __init__(self, num_actions):
    super(QModel, self).__init__()
    self.dropout = nn.Dropout(p=0.3)
    self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding='same')
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding='same')
    self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=1)
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(1152, 512)
    self.fc2 = nn.Linear(512, num_actions)

  def forward(self, x):
    x = nn.functional.relu(self.conv1(x))
    x = nn.functional.relu(self.conv2(x))
    x = self.dropout(x)
    x = nn.functional.relu(self.conv3(x))
    x = self.flatten(x)
    x = nn.functional.relu(self.fc1(x))
    x = self.dropout(x)
    action = self.fc2(x)
    return action


## 모델 빌딩

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# The first model makes the predictions for Q-values which are used to make an action.
model = QModel(num_actions)
model.to(device)

# Build a target model for the prediction of future rewards.
# The weights of a target model get updated every 10000 steps, thus when the loss between the Q-values is calculated, the target Q-value is stable.
model_target = QModel(num_actions)
model_target.to(device)

QModel(
  (dropout): Dropout(p=0.3, inplace=False)
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (conv3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1152, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=64, bias=True)
)

 ## Loss 및 최적화 계산기

In [17]:
loss_function = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00025)

In [18]:
device

device(type='cuda', index=0)

## Replay Buffer 정의

In [19]:
# Experience replay buffers
action_history = []
action_mask_history = []
state_history = []
state_next_history = []
rewards_history = []
done_history = []
episode_reward_history = []
running_reward = 0
episode_count = 0
frame_count = 0


# Number of frames to take random action and observe output
epsilon_random_frames = 50000
# Number of frames for exploration
epsilon_greedy_frames = 200000.0
# Maximum replay length
# Note: The Deepmind paper suggests 1000000 however this causes memory issues
max_memory_length = 500000
# Train the model after 4 actions
update_after_actions = 4
# How often to update the target network
update_target_network = 10000


## 전처리

- env가 리턴하는 observation은 일단 np.array이니 torch.tensor로 캐스팅
- env가 리턴하는 상태가 (8,8,1)의 HWC 이미지 텐서이므로 이를 (3,15,15)의 CHW 이미지로 변환
- One Hot Encoding도 필요

In [20]:
# Funciton to preprocess the state
# Note that player1 = env player, player2 = agent
def preprocess_state(env_observ):
  st = torch.from_numpy(env_observ).squeeze()
  st = st.to(torch.int64)
  st = torch.nn.functional.one_hot(st, num_classes=3)
  st = st.permute(2,0,1)
  return st.to(torch.float32)


## Epsilon-Greedy 액션 선택 함수
학습시 에피소드를 생성하면서 사용 (주의 : 입력은 batch axis 없음)

In [21]:
# Function to select an action
# model : the torch model to compute action-state value (i.e., q-value)
# state : a torch tensor (3x8x8) of float32, which is output by preprocess_state
# mask : a 64-size array (np.array)

def get_greedy_epsilon(model, state, mask):
  global epsilon

  # if frame_count < epsilon_random_frames or np.random.rand(1)[0] < epsilon :
  if np.random.rand(1)[0] < epsilon:
    action = np.random.choice([ i for i in range(num_actions) if mask[i] == 1 ])
  else:
    with torch.no_grad():
      # add a batch axis
      state_tensor = state.unsqueeze(0)
      # compute the q-values
      q_values = model(state_tensor)
      # select the q-values of valid actions
      action = torch.argmax(
          q_values.to('cpu').squeeze() + torch.from_numpy(mask) * 100.,
          # trick to select a valid action
          dim=0 )

      # valid_q = [ (i, q_values[0][i]) for i in range(64) if mask[i] == 1 ]
      # the action of maximum q-value
      # action, _ = max(valid_q, key=lambda e: e[1])


  # decay epsilon
  epsilon -= epsilon_interval / epsilon_greedy_frames
  epsilon = max(epsilon, epsilon_min)

  return action

## Greedy 액션 선택 함수
나중에 evaluation 시 사용

In [22]:
def get_greedy_action(model, state, mask):
  global epsilon

  with torch.no_grad():
    state_tensor = state.unsqueeze(0)  # batch dimension
    q_values = model(state_tensor)

    action = torch.argmax(
        q_values.to('cpu').squeeze() + torch.from_numpy(mask) * 100.,
        # trick to select a valid action
        dim=0 )

  return action

## Update
- Replay buffer에서 batch 하나를 샘플링하고, model을 update한다

In [23]:
# sample a batch of _batch_size from replay buffers
# return numpy.ndarrays

def sample_batch(_batch_size):
  # Get indices of samples for replay buffers
  indices = np.random.choice( range(len(done_history)), size=_batch_size, replace=False )

  state_sample = np.array( [state_history[i].squeeze(0).numpy() for i in indices] )
  state_next_sample = np.array( [state_next_history[i].squeeze(0).numpy() for i in indices] )
  rewards_sample = np.array( [rewards_history[i] for i in indices], dtype=np.float32 )
  action_sample = np.array( [action_history[i] for i in indices] )

  # action mask is the mask for the valid actions at the '''next''' state
  action_mask_sample = np.array( [action_mask_history[i] for i in indices] )
  done_sample = np.array( [float(done_history[i]) for i in indices] )

  return state_sample, state_next_sample, rewards_sample, action_sample, action_mask_sample, done_sample

In [24]:
# Function to update the Q-network
def update_network():
  #sample a batch of ...
  state_sample, state_next_sample, rewards_sample, action_sample, action_mask_sample, done_sample = sample_batch(batch_size)

  # Convert numpy arrays into PyTorch Tensors
  state_sample = torch.tensor(state_sample, dtype=torch.float32).to(device)
  state_next_sample = torch.tensor(state_next_sample, dtype=torch.float32).to(device)
  action_sample = torch.tensor(action_sample, dtype=torch.int64).to(device)
  action_mask_sample = torch.tensor(action_mask_sample, dtype=torch.int64).to(device)
  rewards_sample = torch.tensor(rewards_sample, dtype=torch.float32).to(device)
  done_sample = torch.tensor(done_sample, dtype=torch.float32).to(device)

  # Compute the target Q-values for the states
  with torch.no_grad():
    future_rewards = model_target(state_next_sample)
    # future_rewards = future_rewards.cpu()

    # Compute the q-value for the next state and the action maximizing the q-value
    # Note : the action should be valid (i.e., mask is set to 1)
    max_q_values = torch.max(
        future_rewards + action_mask_sample * 100.,
        # trick to select a valid action
        dim=1 ).values.detach() - 100.

    # Compute the target q-value
    # If the step was final, max_q_values should not be added
    # We assume that the negative return of the opposite player is the return of next step
    # that is, G(t) = r(t+1) - g*r(t+2) + g^2*r(t+3) - g^3*r(t+4) + ...
    target_q_values = rewards_sample + gamma * max_q_values * (1. - done_sample)


  # It's forward propagation! Compute the Q-values for the taken actions
  q_values = model(state_sample)
  # q_values = q_values.cpu()
  q_values_action = q_values.gather(dim=1, index=action_sample.unsqueeze(1)).squeeze(1)


  # Compute the Loss
  loss = loss_function(q_values_action, target_q_values)


  # Perform the Optimization Step
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

# Run DQN Training

In [27]:
for _ in range(max_episodes):
  state, info = env.reset()
  state = preprocess_state(state)
  action_mask = info['action_mask'].reshape((-1,))
  episode_reward = 0

  for timestep in range(1, max_steps_per_episode):
    frame_count += 1

    # Select an action
    # state_cuda = state.to(device)
    action = get_greedy_epsilon(model, state.to(device), action_mask)
    if action < 0:
      print(action_mask)

    # Take the selected action
    state_next, reward, done, info = env.step((action // 8, action % 8))
    state_next = preprocess_state(state_next)
    action_mask = info['action_mask'].reshape((-1,))

    episode_reward += reward

    # Store the transition in the replay buffer
    action_history.append(action)
    action_mask_history.append(action_mask)
    state_history.append(state)
    state_next_history.append(state_next)
    rewards_history.append(reward)
    done_history.append(done)

    state = state_next

    # Update every fourth frame and once batch size is over 32
    if frame_count % update_after_actions == 0 and len(done_history) > batch_size:
      update_network()

    if frame_count % update_target_network == 0:
      model_target.load_state_dict(model.state_dict())

    # Limit the state and reward history
    if len(rewards_history) > max_memory_length:
      del rewards_history[:1]
      del state_history[:1]
      del state_next_history[:1]
      del action_history[:1]
      del action_mask_history[:1]
      del done_history[:1]

    if done:
      break


  episode_count += 1
  episode_reward_history.append(episode_reward)

  # Update running reward to check condition for solving
  if len(episode_reward_history) > 100:
    del episode_reward_history[:1]
  running_reward = np.mean(episode_reward_history)

  if episode_count % 10 == 0:
    print(f'Episode : {episode_count}, Frame count : {frame_count}, Running reward : {running_reward}')

  if episode_count % 5000 == 0:
    torch.save(model, 'model.{}'.format(episode_count))
  # if running_reward > 20:
  #   print(f"Solved at episode {episode_count}!")
  #   break


torch.save(model, 'model.final')


Episode : 10, Frame count : 624, Running reward : -42.3
Episode : 20, Frame count : 1190, Running reward : -39.45
Episode : 30, Frame count : 1780, Running reward : -39.3
Episode : 40, Frame count : 2379, Running reward : -39.45
Episode : 50, Frame count : 2951, Running reward : -39.0
Episode : 60, Frame count : 3527, Running reward : -38.766666666666666
Episode : 70, Frame count : 4106, Running reward : -38.642857142857146
Episode : 80, Frame count : 4699, Running reward : -38.725
Episode : 90, Frame count : 5316, Running reward : -39.05555555555556
Episode : 100, Frame count : 5883, Running reward : -38.82
Episode : 110, Frame count : 6421, Running reward : -37.97
Episode : 120, Frame count : 6981, Running reward : -37.91
Episode : 130, Frame count : 7552, Running reward : -37.72
Episode : 140, Frame count : 8098, Running reward : -37.19
Episode : 150, Frame count : 8662, Running reward : -37.11
Episode : 160, Frame count : 9237, Running reward : -37.1
Episode : 170, Frame count : 98

## Evaluation

In [28]:
import time, sys
from IPython.display import clear_output

In [29]:
board, info = env.reset()
state = preprocess_state(board)
action_mask = info['action_mask'].reshape((-1,))
done = False
env.render()

while not done:
  action = get_greedy_action(model, state.to(device), action_mask)
  print("action : ({}, {})".format(action // 8, action % 8))
  sys.stdout.flush()

  time.sleep(1.0)
  clear_output(wait=True)
  board, reward, done, info = env.step((action // 8, action % 8))
  state = preprocess_state(board)
  action_mask = info['action_mask'].reshape((-1,))
  env.render()

         |         
  M      |         
  H      |   H     
HHHHH M  | HHHHH   
  HM     |   H     
 HHHM    |  HHH    
         |         
         |         

